# Librerias y setup de ngspice

In [ ]:
!sudo apt-get install ngspice libngspice0

# Create a symbolic link for libngspice.so if it doesn't exist
# This is often needed because PySpice looks for libngspice.so (without the version number)
!if [ ! -f /usr/lib/x86_64-linux-gnu/libngspice.so ]; then \
    sudo ln -s /usr/lib/x86_64-linux-gnu/libngspice.so.0 /usr/lib/x86_64-linux-gnu/libngspice.so; \
    echo "Created symlink /usr/lib/x86_64-linux-gnu/libngspice.so"; \
fi

# Set the environment variable for PySpice to find ngspice
import os
os.environ['PYSPICE_NGSPICE_PATH'] = '/usr/lib/x86_64-linux-gnu/libngspice.so.0'

!pip install PySpice

# Diagnostic: Find the actual path of libngspice.so
# Please share the output of this command after execution.
!dpkg -L ngspice
!dpkg -L libngspice0

# Further Diagnostics: Check file existence and dependencies
!ls -l /usr/lib/x86_64-linux-gnu/libngspice.so
!ldd /usr/lib/x86_64-linux-gnu/libngspice.so

import numpy as np
import matplotlib.pyplot as plt
import PySpice.Logging.Logging as Logging
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.Parser import SpiceParser
import re
import torch
import scipy.signal as signal
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import concurrent.futures
import multiprocessing
import random
import warnings
import pandas as pd
import seaborn as sns
from scipy.fft import fft, fftfreq

# Procesador de Netlist (SPICE)

In [ ]:
class NetlistProcessor:
    """
    Motor de análisis de Netlists para el Kaspix Omni-Pipeline.
    Encargado de parsear, validar y extraer la configuración paramétrica
    de circuitos SPICE estándar.
    """

    def __init__(self, file_path):
        self.file_path = file_path
        self.content = ""
        self.params_raw = {}      # Diccionario original (texto, ej: '10k')
        self.params_float = {}    # Diccionario numérico (float, ej: 10000.0)
        self.io_config = {        # Metadatos de entrada/salida
            "input_source": None,
            "output_node": None,
            "ground_check": False
        }
        self.is_parsed = False

        # Carga inmediata
        self._load_file()

    def _load_file(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"[Kaspix Error] No se encuentra el archivo: {self.file_path}")

        with open(self.file_path, 'r', encoding='utf-8') as f:
            self.content = f.read()

    def _spice_to_float(self, value_str):
        """
        Convierte sufijos de ingeniería SPICE a float de Python.
        Soporta: T, G, Meg, k, m, u, n, p, f
        """
        value_str = value_str.lower().strip('{} ') # Limpiar llaves y espacios

        # Mapa de multiplicadores SPICE
        suffixes = {
            't': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3,
            'mil': 25.4e-6, 'm': 1e-3, 'u': 1e-6,
            'n': 1e-9, 'p': 1e-12, 'f': 1e-15
        }

        # Regex para separar número y sufijo
        # Busca un número (int o float) seguido opcionalmente de letras
        match = re.match(r'^([\d\.\-\+]+)([a-z]*)$', value_str)

        if not match:
            try:
                return float(value_str) # Intento directo
            except ValueError:
                return None # No es un número parseable (ej: una fórmula compleja)

        number, suffix = match.groups()
        multiplier = 1.0

        if suffix:
            # SPICE es 'greedy' con los sufijos, 'meg' es especial
            if suffix.startswith('meg'):
                multiplier = suffixes['meg']
            elif suffix[0] in suffixes:
                multiplier = suffixes[suffix[0]]

        return float(number) * multiplier

    def analyze(self):
        """
        Ejecuta el análisis léxico del netlist.
        """
        lines = self.content.splitlines()

        # Regex Compilados
        re_param = re.compile(r'\.param\s+(\w+)\s*=\s*(\{?[\w\.\-\+]+\}?)', re.IGNORECASE)
        re_source = re.compile(r'^\s*([vV]\w+)\s+(\w+)\s+(\w+)', re.IGNORECASE)
        re_save = re.compile(r'\.(?:save|print)\s+(?:v|tran)\s*\((.+?)\)', re.IGNORECASE)

        for line in lines:
            line = line.strip()
            if not line or line.startswith('*'): continue

            # 1. Detectar Parámetros (.param)
            pm = re_param.search(line)
            if pm:
                name, val_str = pm.groups()
                # Guardar raw
                self.params_raw[name] = val_str
                # Convertir a float
                val_float = self._spice_to_float(val_str)
                if val_float is not None:
                    self.params_float[name] = val_float

            # 2. Detectar Inputs (Heurística: Nombre contiene 'in' o 'src')
            sm = re_source.match(line)
            if sm:
                s_name, n_pos, n_neg = sm.groups()
                # Priorizamos fuentes que parezcan de señal
                if 'in' in s_name.lower() or 'src' in s_name.lower() or 'sig' in s_name.lower():
                    self.io_config["input_source"] = s_name

            # 3. Detectar Ground
            if ' 0 ' in line or line.endswith(' 0'):
                self.io_config["ground_check"] = True

            # 4. Detectar Output (.save)
            svm = re_save.search(line)
            if svm:
                # Extraer contenido dentro de paréntesis
                nodes = svm.group(1).replace(')', '').replace('(', '').split()
                if nodes:
                    self.io_config["output_node"] = nodes[0] # Tomamos el primero como principal

        # Fallback para Output si no hay .save
        if not self.io_config["output_node"]:
             if re.search(r'\b(out|output|salida)\b', self.content, re.IGNORECASE):
                 self.io_config["output_node"] = "out (Inferred)"
             else:
                 self.io_config["output_node"] = "UNKNOWN (Define .save V(node))"

        self.is_parsed = True
        return self.get_summary()

    def get_summary(self):
        if not self.is_parsed: return "No analizado."
        return {
            "file_path": self.file_path,
            "valid_spice": self.io_config["ground_check"],
            "knobs_detected": list(self.params_float.keys()),
            "knobs_values": self.params_float,
            "input_source": self.io_config["input_source"],
            "output_target": self.io_config["output_node"]
        }

# Generación de Dataset

In [ ]:
# ============================================================================
# FUNCIONES AUXILIARES - REPRODUCIBILIDAD
# ============================================================================
def set_global_seed(seed=42):
    """
    Establece semillas aleatorias para reproducibilidad total.

    Args:
        seed: Semilla (int)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Si usas CUDA (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Para multi-GPU
        # Configuración para reproducibilidad CUDA (más lento pero determinista)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"✅ Kaspix Seed Locked: {seed}")

# ============================================================
# 1. KASPIX SIGNAL FACTORY (V7: Dynamic Gain + Noise Floor)
# ============================================================
class KaspixSignalFactory:
    def __init__(self, fs, duration_sec, gain_range=(0.1, 1.0)):
        self.fs = fs
        self.duration = duration_sec
        self.n_samples = int(fs * duration_sec)
        self.time_axis = np.linspace(0, duration_sec, self.n_samples)

        # [NUEVO] Guardamos el rango configurado por el usuario
        self.min_v = gain_range[0]
        self.max_v = gain_range[1]

    def _apply_dynamic_gain(self, y):
        """
        Aplica una ganancia aleatoria dentro del rango configurado.
        Usa distribución LOG-UNIFORME para explorar equitativamente
        señales pequeñas (ej. 0.1V) y grandes (ej. 5.0V).
        """
        max_val = np.max(np.abs(y))
        if max_val < 1e-9: return y # Evitar div/0 en silencios

        # Matemáticas para rango logarítmico:
        log_min = np.log(self.min_v)
        log_max = np.log(self.max_v)
        target_peak = np.exp(np.random.uniform(log_min, log_max))

        return (y / max_val) * target_peak

    def _apply_input_noise(self, signal_in):
        # 10% de probabilidad de señal limpia, 90% con algo de ruido realista
        if np.random.rand() > 0.9: return signal_in
        noise = np.random.randn(len(signal_in))
        # SNR aleatorio entre 40dB y 70dB
        noise_level_db = np.random.uniform(-70, -40)
        noise_amplitude = 10 ** (noise_level_db / 20)

        # Escalamos el ruido relativo al pico de la señal actual
        current_peak = np.max(np.abs(signal_in))
        if current_peak == 0: current_peak = 1.0

        return signal_in + (noise * noise_amplitude * current_peak)

    # --- Generadores de Señales (Idénticos, ahora usan self.min_v/max_v implícitamente) ---
    def _pink_noise(self):
        uneven = self.n_samples % 2
        X = np.random.randn(self.n_samples // 2 + 1 + uneven) + 1j * np.random.randn(self.n_samples // 2 + 1 + uneven)
        S = np.sqrt(np.arange(len(X)) + 1.)
        y = (np.fft.irfft(X / S)).real
        if uneven: y = y[:-1]
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _chirp_log(self):
        f_start = 20
        f_end = np.random.uniform(self.fs/4, self.fs/2 * 0.9)
        y = signal.chirp(self.time_axis, f0=f_start, f1=f_end, t1=self.duration, method='logarithmic')
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _step_sequence(self):
        num_steps = np.random.randint(3, 12)
        y = np.zeros_like(self.time_axis)
        indices = np.sort(np.random.choice(np.arange(self.n_samples), num_steps, replace=False))
        levels = np.random.uniform(-1.0, 1.0, num_steps) # Normalizado primero
        current_idx = 0
        for i, idx in enumerate(indices):
            y[current_idx:idx] = levels[i-1] if i > 0 else 0
            current_idx = idx
        y[current_idx:] = levels[-1]
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _multitone(self):
        num_tones = np.random.randint(3, 15)
        y = np.zeros_like(self.time_axis)
        for _ in range(num_tones):
            freq = np.random.uniform(20, self.fs/3)
            phase = np.random.uniform(0, 2*np.pi)
            y += np.sin(2 * np.pi * freq * self.time_axis + phase)
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _impulse_train(self):
        y = np.zeros_like(self.time_axis)
        num_impulses = np.random.randint(1, 5)
        indices = np.random.randint(0, self.n_samples, num_impulses)
        y[indices] = 1.0
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def get_signal(self, recipe):
        keys = list(recipe.keys())
        probs = np.array(list(recipe.values()))
        probs /= probs.sum()
        choice = np.random.choice(keys, p=probs)

        if choice == 'sine':
            f = np.random.uniform(20, 1000)
            raw = np.sin(2*np.pi*f*self.time_axis)
            return self._apply_input_noise(self._apply_dynamic_gain(raw)), f"Sine {int(f)}Hz"

        if choice == 'chirp': return self._chirp_log(), "Chirp Log"
        if choice == 'pink_noise': return self._pink_noise(), "Pink Noise"
        if choice == 'step_sequence': return self._step_sequence(), "Step Seq"
        if choice == 'multitone': return self._multitone(), "Multitone"
        if choice == 'impulse': return self._impulse_train(), "Impulse"
        if choice == 'silence_decay':
             raw = np.zeros_like(self.time_axis)
             return self._apply_input_noise(raw), "Silence Decay" # Ruido de fondo solamente

        return self._pink_noise(), "Default (Pink)"

# ============================================================
# Configuración de Silencio para Logs No-Críticos
logging.getLogger("PySpice").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning) # Calla advertencias de keywords

# ------------------------------------------------------------
# WORKER V8.2 (Estable y Seguro)
# ------------------------------------------------------------
def _simulation_worker(task_payload):
    try:
        # Desempaquetado
        file_path = task_payload['file_path']
        input_source = task_payload['input_source']
        output_target = task_payload['output_target']
        fs = task_payload['fs']
        duration = task_payload['duration']
        input_signal = task_payload['input_signal']
        knob_config = task_payload['knob_config']
        task_id = task_payload['id']

        # --- LÓGICA SPICE ---
        parser = SpiceParser(path=file_path)
        circuit = parser.build_circuit()

        # Eje de tiempo local
        n_samples = len(input_signal)
        time_axis = np.linspace(0, duration, n_samples)

        # Inyección de Fuente (Búsqueda Robusta)
        actual_source_name = None
        for element in circuit.element_names:
            if element.upper() == input_source.upper():
                actual_source_name = element
                break

        if actual_source_name:
            original_source = circuit[actual_source_name]
            # Extraer nodos originales de la fuente detectada
            node_plus = original_source.nodes[0]
            node_minus = original_source.nodes[1]

            # Eliminar fuente original
            circuit._elements.pop(actual_source_name)

            # Preparar datos PWL
            t_list = [float(x) for x in time_axis]
            v_list = [float(x) for x in input_signal]
            input_data = list(zip(t_list, v_list))

            # [CORRECCIÓN CRÍTICA] Sin argumento 'dc_value'
            circuit.PieceWiseLinearVoltageSource(
                'Input_UES',
                node_plus,
                node_minus,
                values=input_data
            )
        else:
            return {"status": "error", "msg": f"Fuente '{input_source}' no encontrada", "id": task_id}

        # Aplicar Knobs
        for name, val in knob_config.items():
            if name == 'dummy_param': continue
            circuit.parameter(name, val)

        # Simular
        simulator = circuit.simulator(temperature=25, nominal_temperature=25)
        # Relajar tolerancias para evitar 'Time step too small' en diodos
        simulator.options('RELTOL=1e-3', 'ABSTOL=1e-9', 'VNTOL=1e-6')

        step_time = 1.0 / fs
        analysis = simulator.transient(step_time=step_time, end_time=duration)

        # Extraer Output
        output_signal = None
        # Búsqueda insensible a mayúsculas para el nodo de salida
        for node_name, node_data in analysis.nodes.items():
            if str(node_name).upper() == str(output_target).upper():
                output_signal = np.array(node_data)
                break

        if output_signal is None:
            available = list(analysis.nodes.keys())
            return {"status": "error", "msg": f"Output '{output_target}' no está. Nodos: {available}", "id": task_id}

        # [DEAD SIGNAL CHECK]
        # Si la entrada tiene energía pero la salida es cero absoluto, es un fallo numérico.
        if np.max(np.abs(output_signal)) < 1e-12 and np.max(np.abs(input_signal)) > 1e-3:
             return {"status": "error", "msg": "Dead Output (Fallo Convergencia)", "id": task_id}

        # Resamplear si es necesario
        if len(output_signal) != len(time_axis):
            output_signal = np.interp(time_axis, np.array(analysis.time), output_signal)

        return {
            "status": "ok",
            "id": task_id,
            "y": output_signal.astype(np.float32),
            "x_meta": {
                "knobs": np.array(list(knob_config.values()), dtype=np.float32),
                "knob_names": list(knob_config.keys())
            }
        }

    except Exception as e:
        return {"status": "error", "msg": str(e), "id": task_payload.get('id', -1)}


# ------------------------------------------------------------
# GENERADOR KASPIX V8 (Anti-Spam)
# ------------------------------------------------------------
class KaspixParallelGenerator:
    def __init__(self, processor_result, fs=48000, duration_sec=0.2, recipe=None, seed=42, signal_range=(0.1, 1.0), param_constraints={}, default_var_pct=0.5):
        self.meta = processor_result
        self.fs = fs
        self.duration = duration_sec
        self.seed = seed
        # Usamos la Factory V7 (con ruido) que definimos antes
        self.factory = KaspixSignalFactory(fs, duration_sec, gain_range=signal_range)
        self.constraints = param_constraints
        self.default_var = default_var_pct
        self.time_axis = self.factory.time_axis
        self.recipe = recipe if recipe else {"chirp": 1.0}
        self.max_workers = 24 # Ajusta según tus núcleos

    def _sample_knobs(self):
        """
        Genera una configuración de knobs híbrida:
        1. Prioridad: Rangos Absolutos definidos en PARAM_CONSTRAINTS.
        2. Fallback: Variación relativa (±%) sobre el valor nominal del netlist.
        """
        nominal = self.meta['knobs_values'] # Valores extraídos del .net
        if not nominal: return {'dummy_param': 1.0}

        sampled = {}
        for name, val in nominal.items():
            if name in self.constraints: # CASO A: Tenemos una regla explícita para este parámetro (ej. Potenciómetros)
                min_v, max_v = self.constraints[name]
                new_val = np.random.uniform(min_v, max_v)
            else: # CASO B: Es un componente desconocido/fijo, variamos alrededor del nominal
                delta = val * self.default_var
                # Evitamos valores negativos o cero si no son deseados
                low_bound = val - delta
                if low_bound <= 0: low_bound = val * 0.01 # Floor seguro

                new_val = np.random.uniform(low_bound, val + delta)
            sampled[name] = new_val
        return sampled

    def build_dataset(self, n_samples=100, save_path="kaspix_dataset.pt"):
        set_global_seed(self.seed)
        print(f"--- Preparando {n_samples} tareas ---")

        tasks = []
        pre_gen = {}

        for i in range(n_samples):
            sig_in, sig_type = self.factory.get_signal(self.recipe)
            knobs = self._sample_knobs()

            pre_gen[i] = {"audio_in": sig_in.astype(np.float32), "signal_type": sig_type}

            tasks.append({
                "id": i,
                "file_path": self.meta['file_path'],
                "input_source": self.meta['input_source'],
                "output_target": self.meta['output_target'],
                "fs": self.fs,
                "duration": self.duration,
                "input_signal": sig_in,
                "knob_config": knobs
            })

        print(f" Procesando...")

        dataset_x = []
        dataset_y = []
        errors = []

        with concurrent.futures.ProcessPoolExecutor(max_workers=self.max_workers) as executor:
            # Barra de progreso limpia
            results = list(tqdm(executor.map(_simulation_worker, tasks), total=n_samples))

        # Ordenar resultados
        results.sort(key=lambda x: x['id'])

        for res in results:
            if res['status'] == 'ok':
                idx = res['id']
                in_meta = pre_gen[idx]
                wk_meta = res['x_meta']

                dataset_x.append({
                    "audio_in": in_meta['audio_in'],
                    "signal_type": in_meta['signal_type'],
                    "knobs": wk_meta['knobs'],
                    "knob_names": wk_meta['knob_names']
                })
                dataset_y.append(res['y'])
            else:
                errors.append(res['msg'])

        success_count = len(dataset_x)

        if success_count == 0:
            print("\n FALLO TOTAL. Primer error detectado:")
            print(f"   >> {errors[0] if errors else 'Desconocido'}")
            return [], []

        if len(errors) > 0:
            print(f"\n Advertencia: {len(errors)} muestras fallaron (probablemente inestabilidad numérica).")
            print(f"   Ejemplo de error: {errors[0]}")
            print(f"   Total exitoso: {success_count}/{n_samples}")

        print(f"Guardando {success_count} muestras válidas en {save_path}...")
        torch.save({
            "x": dataset_x, "y": dataset_y, "fs": self.fs,
            "meta": self.meta, "recipe": self.recipe, "seed": self.seed
        }, save_path)

        return dataset_x, dataset_y

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ejecución

In [ ]:
# ============================================================
# BLOQUE PRINCIPAL DE EJECUCIÓN (KASPIX OMNI-PIPELINE V4)
# ============================================================
if __name__ == "__main__":
    # 1. DEFINICIÓN DE OBJETIVOS Y ESTRATEGIA
    # ------------------------------------------------------------
    TARGET_NETLIST = "diode.cir"
    OUTPUT_DATASET = "diode_data.pt"

    # Configuración de Simulación
    N_SAMPLES = 20       # Aumentamos un poco para ver variedad estadística
    SAMPLE_RATE = 48000  # Hz
    DURATION = 0.02       # Segundos
    SIGNAL_LEVEL_RANGE = (0.1, 3.0) # Volts
    PARAM_CONSTRAINTS = {
    # Suponiendo que tu netlist tiene params llamados 'vol' o 'drive' para potenciómetros
    "vol": (0.0, 1.0),      # Barrer todo el recorrido
    "drive": (0.0, 1.0),    # Barrer todo el recorrido
    "tone": (0.0, 1.0),
    # "C1": (1e-9, 100e-9) # Ejemplo: Si quisieras "modear" el circuito cambiando un capacitor clave
    }

    DEFAULT_VARIATION_PCT = 0.5 # Para los params que NO estén en la lista de arriba, ¿cuánto permitimos que varíen respecto a su valor original?

    # RECETA DE EXCITACIÓN (Domain Adaptation Strategy)
    RECIPE_MIX = {
        "chirp": 0.25,          # Respuesta en Frecuencia (Bode)
        "pink_noise": 0.20,     # Dinámica Espectral Compleja
        "multitone": 0.15,      # Intermodulación
        "step_sequence": 0.20,  # Transitorios / Settling Time
        "sine": 0.10,           # Pura frecuencia
        "impulse": 0.05,        # Respuesta al Impulso
        "silence_decay": 0.05   # Piso de ruido y descarga
    }

    print(f"INICIANDO KASPIX OMNI-PIPELINE (V4 - Signal Factory)")
    print(f"Objetivo Físico: {TARGET_NETLIST}")
    print(f"Fn Receta de Señales: {RECIPE_MIX}")

    # 2. FASE 1: ANÁLISIS DE TOPOLOGÍA (NetlistProcessor)
    # ------------------------------------------------------------
    print("\n[Fase 1] Analizando Física del Netlist...")
    # Asegurarse de tener el archivo (crearlo si no existe para la demo)
    if not os.path.exists(TARGET_NETLIST):
        print("ERROR No hay archivo .cir")

    processor = NetlistProcessor(TARGET_NETLIST)
    circuit_meta = processor.analyze()

    # Validación Estricta
    if not circuit_meta['valid_spice']:
        print("CRITICAL: El Netlist no es válido (GND 0 missing?).")
        exit(1)

    print("✅ Netlist Interpretado Correctamente.")
    print(f"   -> Knobs: {list(circuit_meta['knobs_values'].keys())}")
    print(f"   -> I/O: {circuit_meta['input_source']} -> {circuit_meta['output_target']}")

    # 3. FASE 2: GENERACIÓN MASIVA (KaspixParallelGenerator)
    # ------------------------------------------------------------
    print(f"\n[Fase 2] Generando {N_SAMPLES} simulaciones con Variabilidad Estocástica...")

    # Instanciamos con la RECETA
    generator = KaspixParallelGenerator(
      circuit_meta,
      fs=SAMPLE_RATE,
      duration_sec=DURATION,
      recipe=RECIPE_MIX,
      seed=42,
      signal_range=SIGNAL_LEVEL_RANGE,  # 1. Control de Voltaje (Saturación)
      param_constraints=PARAM_CONSTRAINTS, # 2. Control de Parámetros (Knobs vs Tolerancias)
      default_var_pct=DEFAULT_VARIATION_PCT
    )

    data_x, data_y = generator.build_dataset(
        n_samples=N_SAMPLES,
        save_path=OUTPUT_DATASET
    )

    # 4. FASE 3: CONTROL DE CALIDAD VISUAL (QC)
    # ------------------------------------------------------------
    # Si generamos 0 muestras por error, evitar crash
    if len(data_x) == 0:
        print("Error: No se generaron datos.")
        exit(1)

    print("\n[Fase 3] Control de Calidad Visual (Inputs Diversos)...")

    # --- FIX: Generar eje de tiempo manualmente ---
    # Como el generador paralelo no expuso time_axis, lo calculamos aquí
    # usando los parámetros globales definidos al inicio.
    time_axis = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))
    # ----------------------------------------------

    indices_to_plot = [0, N_SAMPLES // 2, N_SAMPLES - 1]
    indices_to_plot = [i for i in indices_to_plot if i < len(data_x)]

    plt.figure(figsize=(12, 10))

    # Subplot 1: Señales de Excitación
    ax1 = plt.subplot(2, 1, 1)
    ax1.set_title("Inputs Generados (Signal Factory)")

    for idx in indices_to_plot:
        sig_type = data_x[idx]['signal_type']
        ax1.plot(
            time_axis * 1000,    # <--- USAMOS LA VARIABLE LOCAL
            data_x[idx]['audio_in'],
            alpha=0.7,
            label=f"#{idx}: {sig_type}"
        )
    ax1.set_ylabel("Voltaje (V)")
    ax1.legend(loc='upper right', fontsize='small')
    ax1.grid(True, alpha=0.3)

    # Subplot 2: Respuestas Físicas
    ax2 = plt.subplot(2, 1, 2)
    # Safely get knobs for title
    sample_knobs = list(data_x[0]['knob_names']) if data_x else []
    ax2.set_title(f"Respuestas del Sistema (Knobs: {sample_knobs})")

    for idx in indices_to_plot:
        knob_vals = data_x[idx]['knobs']
        sig_type = data_x[idx]['signal_type']
        k_names = data_x[idx]['knob_names']

        params_str = " | ".join([f"{n}:{v:.1e}" for n, v in zip(k_names, knob_vals)])

        ax2.plot(
            time_axis * 1000,   # <--- USAMOS LA VARIABLE LOCAL
            data_y[idx],
            label=f"#{idx} [{sig_type}]: {params_str}"
        )

    ax2.set_xlabel("Tiempo (ms)")
    ax2.set_ylabel("Voltaje (V)")
    ax2.legend(loc='upper right', fontsize='x-small')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\n KASPIX PIPELINE COMPLETADO.")
    print(f"   Dataset guardado en: {OUTPUT_DATASET}")

# Como cargar los datos (FiLM) y Estadística de parámetros


In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class KaspixDatasetFiLM(Dataset):
    def __init__(self, pt_file_path):
        print(f"📂 [Kaspix IO] Cargando Dataset FiLM desde: {pt_file_path}")

        try:
            loaded = torch.load(pt_file_path, weights_only=False)
        except FileNotFoundError:
            raise FileNotFoundError(f"❌ No se encontró el archivo: {pt_file_path}")

        self.data_x = loaded['x']
        self.data_y = loaded['y']
        self.meta = loaded['meta']

        # Extracción de nombres de parámetros para reporte
        # Asumimos que todas las muestras tienen la misma estructura
        self.knob_names = self.data_x[0]['knob_names']

        # 2. Vectorización de Knobs (Para cálculo rápido)
        # Convertimos la lista de listas en una matriz NumPy [N_Muestras, N_Knobs]
        all_knobs = np.array([item['knobs'] for item in self.data_x])

        # 3. Cálculo de Tensores de Normalización (Min-Max)
        self.k_min = torch.from_numpy(all_knobs.min(axis=0)).float()
        self.k_max = torch.from_numpy(all_knobs.max(axis=0)).float()

        # Protección: Si min == max (parametro fijo), evitamos división por cero
        # Sumamos un epsilon al max para que la normalización sea 0.0
        self.k_max[self.k_max == self.k_min] += 1e-6

        # 4. DASHBOARD ESTADÍSTICO
        print(f"\n📊 ANÁLISIS DE PARÁMETROS FÍSICOS ({len(self.data_x)} muestras)")
        print(f"{'='*75}")
        print(f"{'ID':<4} | {'Nombre (.param)':<20} | {'Min':<10} | {'Max':<10} | {'Media':<10} | {'Std Dev':<10}")
        print(f"{'-'*75}")

        for i, name in enumerate(self.knob_names):
            vals = all_knobs[:, i]
            v_min = vals.min()
            v_max = vals.max()
            v_mean = vals.mean()
            v_std = vals.std()

            print(f"{i:<4} | {name:<20} | {v_min:<10.4f} | {v_max:<10.4f} | {v_mean:<10.4f} | {v_std:<10.4f}")

        print(f"{'='*75}\n")
        print("✅ Dataset listo para entrenamiento.")

    def __len__(self):
        return len(self.data_x)

    def __getitem__(self, idx):
        # A. Audio Input [1, L]
        audio_raw = self.data_x[idx]['audio_in']
        x_audio = torch.from_numpy(audio_raw).float().unsqueeze(0)

        # B. Knobs Input [N_Params] (Normalizado 0-1)
        knobs_raw = self.data_x[idx]['knobs']
        x_knobs = torch.from_numpy(knobs_raw).float()
        x_knobs_norm = (x_knobs - self.k_min) / (self.k_max - self.k_min)

        # C. Target [1, L]
        target_raw = self.data_y[idx]
        y_target = torch.from_numpy(target_raw).float().unsqueeze(0)

        return x_audio, x_knobs_norm, y_target

# ============================================================
# BLOQUE DE VERIFICACIÓN
# ============================================================
if __name__ == "__main__":
    # Nombre del archivo
    DATASET_FILE = OUTPUT_DATASET

    try:
        ds = KaspixDatasetFiLM(DATASET_FILE)

        # Prueba de DataLoader
        dl = DataLoader(ds, batch_size=8, shuffle=True)
        b_audio, b_knobs, b_target = next(iter(dl))

        print("🔎 INSPECCIÓN DE TENSOR (Batch de prueba):")
        print(f"   -> Audio In:  {b_audio.shape}  (Batch, Channels, Time)")
        print(f"   -> Knobs:     {b_knobs.shape}      (Batch, Params)")
        print(f"   -> Target:    {b_target.shape}  (Batch, Channels, Time)")

        # Verificación de Normalización en el Batch
        print("\n🧪 Test de Normalización (Debe ser True):")
        in_range = (b_knobs.max() <= 1.0001) and (b_knobs.min() >= -0.0001)
        print(f"   -> ¿Knobs entre 0 y 1? {'✅ SÍ' if in_range else '❌ NO (Revisar k_max)'}")

    except Exception as e:
        print(f"\n⚠️ Error durante la prueba: {e}")
        print("   (Asegúrate de haber generado el archivo .pt primero)")

# Análisis de los datos (FILTRO PASA BAJO)

In [ ]:
class KaspixAnalytics:
    """
    Módulo de Inteligencia de Datos Kaspix V2.
    Incluye Reconstrucción de Bode (Magnitude Response) vía FFT.
    """
    def __init__(self, dataset_path=OUTPUT_DATASET):
        print(f" Cargando Dataset para Bode Analysis: {dataset_path} ...")
        self.data = torch.load(dataset_path, weights_only=False)
        self.fs = self.data['fs']
        self.samples_x = self.data['x']
        self.samples_y = self.data['y']

        self.df = self._build_dataframe()

    def _build_dataframe(self):
        records = []
        for i, (x, y) in enumerate(zip(self.samples_x, self.samples_y)):
            knob_dict = dict(zip(x['knob_names'], x['knobs']))

            # Heurística para hallar R y C
            r_val = next((v for k, v in knob_dict.items() if 'R' in k.upper()), None)
            c_val = next((v for k, v in knob_dict.items() if 'C' in k.upper()), None)

            fc_theoretical = np.nan
            if r_val and c_val:
                fc_theoretical = 1.0 / (2.0 * np.pi * r_val * c_val)

            records.append({
                "id": i,
                "signal_type": x['signal_type'].split(" ")[0].lower(),
                "fc_Hz": fc_theoretical,
                "R_Ohm": r_val
            })
        return pd.DataFrame(records)

    def _compute_bode(self, x_in, y_out):
        """Calcula H(f) = Y(f)/X(f) para una sola muestra."""
        N = len(x_in)
        # Ventana de Hanning para suavizar bordes espectrales
        window = np.hanning(N)

        # FFT
        X_f = fft(x_in * window)
        Y_f = fft(y_out * window)

        # Evitar división por cero
        with np.errstate(divide='ignore', invalid='ignore'):
            H_f = np.abs(Y_f) / (np.abs(X_f) + 1e-10)
            H_db = 20 * np.log10(H_f)

        freqs = fftfreq(N, 1/self.fs)

        # Retornamos solo la mitad positiva del espectro
        half_n = N // 2
        return freqs[:half_n], H_db[:half_n]

    def generate_report(self):
        df = self.df
        plt.style.use('seaborn-v0_8-whitegrid')

        # Layout: 2 filas. Arriba estadísticas, Abajo EL BODE GIGANTE.
        fig = plt.figure(figsize=(15, 12))
        gs = fig.add_gridspec(3, 2)

        # --- PANEL 1: Distribución fc ---
        ax1 = fig.add_subplot(gs[0, 0])
        if not df['fc_Hz'].isna().all():
            log_fc = np.log10(df['fc_Hz'].dropna())
            sns.histplot(log_fc, kde=True, ax=ax1, color='teal', bins=20)
            ax1.set_title("Distribución de Frecuencias de Corte (Target Físico)")
            ax1.set_xlabel("Log10(Freq Hz)")

        # --- PANEL 2: Tipos de Señal ---
        ax2 = fig.add_subplot(gs[0, 1])
        # Modified to address FutureWarning
        sns.countplot(y='signal_type', hue='signal_type', data=df, ax=ax2, palette='magma', legend=False)
        ax2.set_title("Receta de Entrenamiento")

        # --- PANEL 3 (GRANDE): BODE PLOT RECONSTRUIDO ---
        ax_bode = fig.add_subplot(gs[1:, :]) # Ocupa toda la parte inferior

        print(" Calculando Bode Plots estimados (System Identification)...")

        # Filtramos solo señales "Broadband" (Chirp o Noise)
        # Las senoidales puras no sirven para ver todo el espectro.
        broadband_indices = df[df['signal_type'].isin(['chirp', 'pink', 'white'])].index

        if len(broadband_indices) > 0:
            colors = plt.cm.viridis(np.linspace(0, 1, len(broadband_indices)))

            # Graficar líneas individuales (transparentes)
            for idx, color in zip(broadband_indices[:50], colors): # Max 50 para no matar la gráfica
                x_sig = self.samples_x[idx]['audio_in']
                y_sig = self.samples_y[idx]
                fc_val = df.loc[idx, 'fc_Hz']

                freqs, mag_db = self._compute_bode(x_sig, y_sig)

                label_txt = f"fc={fc_val:.0f}Hz" if not np.isnan(fc_val) else None
                ax_bode.semilogx(freqs, mag_db, color=color, alpha=0.3, linewidth=1)

            # Estética del Bode
            ax_bode.set_title(f"Diagrama de Bode Reconstruido (N={len(broadband_indices)} muestras Broadband)", fontsize=14)
            ax_bode.set_xlabel("Frecuencia (Hz) [Escala Log]", fontsize=12)
            ax_bode.set_ylabel("Magnitud (dB)", fontsize=12)
            ax_bode.set_xlim(20, self.fs/2) # De 20Hz a Nyquist
            ax_bode.set_ylim(-60, 10)       # Rango dinámico típico
            ax_bode.grid(True, which="both", ls="-", alpha=0.5)

            # Línea de referencia de -3dB
            ax_bode.axhline(-3, color='red', linestyle='--', linewidth=1.5, label="-3 dB (Corte)")
            ax_bode.legend(loc='upper right')

            # Anotación Técnica
            ax_bode.text(0.02, 0.05,
                         "Nota: Estimación H(f) = FFT(out)/FFT(in) usando solo muestras Chirp/Pink Noise.",
                         transform=ax_bode.transAxes, fontsize=9, bbox=dict(facecolor='white', alpha=0.8))
        else:
            ax_bode.text(0.5, 0.5, " No hay señales de banda ancha (Chirp/Noise) para estimar Bode.",
                         ha='center', fontsize=14)

        plt.tight_layout()
        plt.show()

# ==========================================
# EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    analytics = KaspixAnalytics(OUTPUT_DATASET)
    analytics.generate_report()